# 🦶 Toe Be Sure — DFU Model Training
Trains ResNet18 on the DFU dataset with:
- Class imbalance handling (WeightedRandomSampler)
- Cosine LR annealing + early stopping
- 70 / 15 / 15 train / val / test split
- Final test metrics: Accuracy, AUC, F1, Sensitivity, Specificity

**Run cells top to bottom in Google Colab (GPU runtime recommended).**

## ⚡ Quick Accuracy Check (no training needed)
Run **only** this cell to evaluate the existing `best_dfu.pt` model against your Drive dataset. No GPU required.

In [ ]:
# ── QUICK EVAL: download model + evaluate on your Drive dataset ───────────
# Prerequisites: Runtime → Connect  (CPU is fine, GPU is faster)
#                Mount Google Drive when prompted

# 1. Install deps
!pip -q install torch torchvision scikit-learn
!pip -q install git+https://github.com/jacobgil/pytorch-grad-cam

# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Download the trained model from GitHub
import os
MODEL_PATH = "/content/best_dfu.pt"
if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress -O "$MODEL_PATH" \
        "https://github.com/kharsha006/hackathon_dfu/raw/main/models/best_dfu.pt"
print(f"Model: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB")

# 4. Find your dataset on Drive
import os
def find_dataset(root):
    for dp, dns, _ in os.walk(root):
        if "Abnormal(Ulcer)" in dns and "Normal(Healthy skin)" in dns:
            return dp
    return None

DATA_DIR = find_dataset("/content/drive/MyDrive/hackathon")
if DATA_DIR is None:
    # Try common Drive locations
    for path in ["/content/drive/MyDrive", "/content/drive/MyDrive/DFU",
                 "/content/drive/MyDrive/hackathon/dfu_data"]:
        DATA_DIR = find_dataset(path)
        if DATA_DIR: break

print("Dataset:", DATA_DIR)
print("Classes:", os.listdir(DATA_DIR) if DATA_DIR else "NOT FOUND — check path above")

# 5. Load model
import torch, torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                              confusion_matrix, classification_report)

device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["Abnormal(Ulcer)", "Normal(Healthy skin)"]

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device).eval()
print("Model loaded on", device)

# 6. Evaluate on 20% random split (reproducible)
import random
tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

full = datasets.ImageFolder(DATA_DIR, transform=tfm)
n = len(full)
idx = list(range(n))
random.seed(42); random.shuffle(idx)
test_idx = idx[int(0.85*n):]   # last 15% as test set

class SubsetDS(torch.utils.data.Dataset):
    def __init__(self, ds, idxs):
        self.ds, self.idxs = ds, idxs
    def __len__(self): return len(self.idxs)
    def __getitem__(self, i): return self.ds[self.idxs[i]]

test_loader = DataLoader(SubsetDS(full, test_idx), batch_size=32,
                         shuffle=False, num_workers=2)

all_probs, all_preds, all_y = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        probs = torch.softmax(model(x), dim=1)
        all_probs.append(probs[:,0].cpu().numpy())
        all_preds.append(probs.argmax(1).cpu().numpy())
        all_y.append(y.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.concatenate(all_preds)
all_y     = np.concatenate(all_y)

binary_y     = (all_y == 0).astype(int)
binary_preds = (all_preds == 0).astype(int)
tn, fp, fn, tp = confusion_matrix(binary_y, binary_preds).ravel()

print("\n" + "="*52)
print("         ACCURACY REPORT — best_dfu.pt")
print("="*52)
print(f"  Test samples  : {len(all_y)}")
print(f"  Accuracy      : {accuracy_score(all_y, all_preds)*100:.2f}%")
print(f"  AUC-ROC       : {roc_auc_score(binary_y, all_probs):.4f}")
print(f"  F1 Score      : {f1_score(binary_y, binary_preds):.4f}")
print(f"  Sensitivity   : {tp/(tp+fn)*100:.2f}%  (ulcer correctly detected)")
print(f"  Specificity   : {tn/(tn+fp)*100:.2f}%  (healthy correctly rejected)")
print("="*52)
print(classification_report(all_y, all_preds, target_names=CLASS_NAMES))

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install matplotlib pillow opencv-python scikit-learn
!pip -q install git+https://github.com/jacobgil/pytorch-grad-cam
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Mount Drive & unzip dataset ──────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/hackathon/DFU.zip"
OUT_DIR  = "/content/dfu_data"
os.makedirs(OUT_DIR, exist_ok=True)

!unzip -q "$ZIP_PATH" -d "$OUT_DIR"
print("Unzipped to:", OUT_DIR)
print("Contents:", os.listdir(OUT_DIR))

In [ ]:
# ── Cell 3: Find dataset root ─────────────────────────────────────────────
import os

def find_dataset_root(root="/content/dfu_data"):
    for dirpath, dirnames, _ in os.walk(root):
        if "Abnormal(Ulcer)" in dirnames and "Normal(Healthy skin)" in dirnames:
            return dirpath
    return None

DATA_DIR = find_dataset_root()
print("Dataset root:", DATA_DIR)
print("Classes:", os.listdir(DATA_DIR))

In [ ]:
# ── Cell 4: Build datasets with 70/15/15 split ───────────────────────────
import random, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Transforms ─────────────────────────────────────────────────────────────
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Full dataset (no transform yet for split)
full = datasets.ImageFolder(DATA_DIR)
class_names = full.classes
n = len(full)
print(f"Classes: {class_names} | Total images: {n}")

# Count per class
class_counts = np.bincount([s[1] for s in full.samples])
print(f"Class counts: { {c:int(class_counts[i]) for i,c in enumerate(class_names)} }")

# 70 / 15 / 15 split
idx = list(range(n))
random.shuffle(idx)
t = int(0.70 * n);  v = int(0.15 * n)
train_idx, val_idx, test_idx = idx[:t], idx[t:t+v], idx[t+v:]

# Apply transforms via wrapper
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, ds, idxs, tfm):
        self.ds, self.idxs, self.tfm = ds, idxs, tfm
    def __len__(self): return len(self.idxs)
    def __getitem__(self, i):
        img, lbl = self.ds.imgs[self.idxs[i]]
        from PIL import Image
        img = Image.open(img).convert("RGB")
        return self.tfm(img), lbl

train_ds = TransformSubset(full, train_idx, train_tfms)
val_ds   = TransformSubset(full, val_idx,   eval_tfms)
test_ds  = TransformSubset(full, test_idx,  eval_tfms)

# WeightedRandomSampler to handle class imbalance
train_labels = [full.imgs[i][1] for i in train_idx]
class_weights = 1.0 / class_counts
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,   num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False,   num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
# ── Cell 5: Build model, optimizer, scheduler ────────────────────────────
from torchvision import models
import torch.nn as nn

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

# Class-weighted loss (extra guard against imbalance)
cw = torch.tensor(class_weights / class_weights.sum(), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=cw)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)

EPOCHS = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"ResNet18 ready | {EPOCHS} epochs | CosineAnnealingLR | WeightedLoss")

In [ ]:
# ── Cell 6: Training loop with early stopping ────────────────────────────
from sklearn.metrics import roc_auc_score
import numpy as np

best_auc   = -1
best_path  = "/content/best_dfu.pt"
patience   = 7
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()

    # ── Validate ──
    model.eval()
    all_probs, all_y = [], []
    correct, total   = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            probs  = torch.softmax(model(x), dim=1)
            pred   = probs.argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
            all_probs.append(probs[:, 0].cpu().numpy())
            all_y.append(y.cpu().numpy())

    val_acc  = correct / total
    all_probs = np.concatenate(all_probs)
    all_y     = np.concatenate(all_y)

    try:
        val_auc = roc_auc_score((all_y == 0).astype(int), all_probs)
    except:
        val_auc = 0.0

    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss/len(train_loader):.4f} "
          f"| val_acc={val_acc:.3f} | val_auc={val_auc:.3f} | lr={lr_now:.2e}")

    if val_auc > best_auc:
        best_auc   = val_auc
        no_improve = 0
        torch.save(model.state_dict(), best_path)
        print(f"  ✅ New best AUC={best_auc:.4f} — saved to {best_path}")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"  ⏹  Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

print(f"\n🏁 Training complete. Best Val AUC: {best_auc:.4f}")

In [ ]:
# ── Cell 7: Final evaluation on held-out TEST set ────────────────────────
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                             confusion_matrix, classification_report)
import numpy as np

# Load best checkpoint
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

all_probs, all_preds, all_y = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        probs = torch.softmax(model(x), dim=1)
        pred  = probs.argmax(dim=1)
        all_probs.append(probs[:, 0].cpu().numpy())
        all_preds.append(pred.cpu().numpy())
        all_y.append(y.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.concatenate(all_preds)
all_y     = np.concatenate(all_y)

binary_y     = (all_y == 0).astype(int)   # 1 = ulcer, 0 = healthy
binary_preds = (all_preds == 0).astype(int)

auc  = roc_auc_score(binary_y, all_probs)
acc  = accuracy_score(all_y, all_preds)
f1   = f1_score(binary_y, binary_preds)
cm   = confusion_matrix(binary_y, binary_preds)

tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)   # recall for ulcer class
specificity = tn / (tn + fp)   # recall for healthy class

print("=" * 50)
print("       TEST SET RESULTS")
print("=" * 50)
print(f"  Accuracy    : {acc*100:.2f}%")
print(f"  AUC-ROC     : {auc:.4f}")
print(f"  F1 Score    : {f1:.4f}")
print(f"  Sensitivity : {sensitivity*100:.2f}%  (ulcer detection rate)")
print(f"  Specificity : {specificity*100:.2f}%  (healthy correctly rejected)")
print("=" * 50)
print("\nDetailed report:")
print(classification_report(all_y, all_preds, target_names=class_names))

In [ ]:
# ── Cell 8: Save model to Google Drive & upload to GitHub ────────────────
import shutil, os, subprocess

# Save to Drive
dst = "/content/drive/MyDrive/hackathon/best_dfu.pt"
shutil.copy(best_path, dst)
print(f"✅ Saved to Drive: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

# Upload to GitHub (replaces the model used by the Vercel app)
GITHUB_REPO = "https://Vishnu-1526:$(cat /content/drive/MyDrive/hackathon/gh_token.txt)@github.com/kharsha006/hackathon_dfu.git"
# Simpler: just download the file manually to your laptop and push via git
# OR store a GitHub token in Drive at /content/drive/MyDrive/hackathon/gh_token.txt

!git clone "$GITHUB_REPO" /content/hackathon_dfu 2>&1 | tail -2
!cp "$dst" /content/hackathon_dfu/models/best_dfu.pt
!cd /content/hackathon_dfu && git config user.email "train@tbs" && git config user.name "TBS Train"
!cd /content/hackathon_dfu && git add models/best_dfu.pt && git commit -m "model: retrain with improved accuracy" && git push
print("✅ Model pushed to GitHub")